In [5]:
# check

import pandas as pd

bls    = pd.read_csv('/Users/av/Desktop/255/255-AI/data/processed/bls/bls_cleaned.csv',    dtype={'OCC_CODE': str})
onet   = pd.read_csv('/Users/av/Desktop/255/255-AI/data/processed/onet/onet_cleaned.csv',   dtype={'occ_code': str})
felten = pd.read_csv('/Users/av/Desktop/255/255-AI/data/processed/felten/felten_cleaned.csv', dtype={'OCC_CODE': str})

onet = onet.rename(columns={'occ_code': 'OCC_CODE'})

bls_occs    = set(bls['OCC_CODE'])
onet_occs   = set(onet['OCC_CODE'])
felten_occs = set(felten['OCC_CODE'])

all_three = bls_occs & onet_occs & felten_occs

print(f"BLS occupations:    {len(bls_occs)}")
print(f"O*NET occupations:  {len(onet_occs)}")
print(f"Felten occupations: {len(felten_occs)}")
print(f"All three overlap:  {len(all_three)}")

BLS occupations:    750
O*NET occupations:  774
Felten occupations: 774
All three overlap:  655


In [10]:
import pandas as pd

# ── Load all three cleaned files ────────────────────────────────
bls    = pd.read_csv('/Users/av/Desktop/255/255-AI/data/processed/bls/bls_cleaned.csv',    dtype={'OCC_CODE': str})
onet   = pd.read_csv('/Users/av/Desktop/255/255-AI/data/processed/onet/onet_cleaned.csv',   dtype={'occ_code': str})
felten = pd.read_csv('/Users/av/Desktop/255/255-AI/data/processed/felten/felten_cleaned.csv', dtype={'OCC_CODE': str})

# ── Standardize join key name ────────────────────────────────────
onet = onet.rename(columns={'occ_code': 'OCC_CODE'})

# ── Merge ────────────────────────────────────────────────────────
master = onet.merge(felten, on='OCC_CODE', how='inner')
print(f"After O*NET + Felten: {len(master)} occupations")

master = master.merge(bls, on='OCC_CODE', how='inner')
print(f"After + BLS:          {len(master)} occupations")

# ── Drop redundant title column from Felten ──────────────────────
# Keep OCC_TITLE from O*NET, drop Felten's version
if 'OCC_TITLE_FELTEN' in master.columns:
    master = master.drop(columns=['OCC_TITLE_FELTEN'])

# ── Final checks ─────────────────────────────────────────────────
print(f"\nFinal shape:      {master.shape}")
print(f"Unique SOC codes: {master['OCC_CODE'].nunique()}")
print(f"Missing values:   {master.isnull().sum().sum()}")
print(f"\nColumns: {master.columns.tolist()}")

# ── Spot checks ──────────────────────────────────────────────────
print("\nSpot check — Chief Executives (11-1011):")
print(master[master['OCC_CODE'] == '11-1011']
      [['OCC_CODE', 'OCC_TITLE', 'AIOE',
        'TOT_EMP_2019', 'TOT_EMP_2024',
        'real_median_wage_2019', 'real_median_wage_2024',
        'emp_change_pct_19_24', 'wage_change_pct_19_24']].to_string())

print("\nSpot check — Registered Nurses (29-1141):")
print(master[master['OCC_CODE'] == '29-1141']
      [['OCC_CODE', 'OCC_TITLE', 'AIOE',
        'TOT_EMP_2019', 'TOT_EMP_2024',
        'emp_change_pct_19_24', 'wage_change_pct_19_24']].to_string())

# ── Save ─────────────────────────────────────────────────────────
master.to_csv('data/processed/master_occupations.csv', index=False)
print(f"\nSaved: data/processed/master_occupations.csv")

After O*NET + Felten: 682 occupations
After + BLS:          655 occupations

Final shape:      (655, 123)
Unique SOC codes: 655
Missing values:   68

Columns: ['OCC_CODE', 'arm_hand_steadiness', 'auditory_attention', 'category_flexibility', 'control_precision', 'deductive_reasoning', 'depth_perception', 'dynamic_flexibility', 'dynamic_strength', 'explosive_strength', 'extent_flexibility', 'far_vision', 'finger_dexterity', 'flexibility_of_closure', 'fluency_of_ideas', 'glare_sensitivity', 'gross_body_coordination', 'gross_body_equilibrium', 'hearing_sensitivity', 'inductive_reasoning', 'information_ordering', 'manual_dexterity', 'mathematical_reasoning', 'memorization', 'multilimb_coordination', 'near_vision', 'night_vision', 'number_facility', 'oral_comprehension', 'oral_expression', 'originality', 'perceptual_speed', 'peripheral_vision', 'problem_sensitivity', 'rate_control', 'reaction_time', 'response_orientation', 'selective_attention', 'sound_localization', 'spatial_orientation', '

KeyError: "['OCC_TITLE'] not in index"

In [11]:
print(master.columns.tolist())

['OCC_CODE', 'arm_hand_steadiness', 'auditory_attention', 'category_flexibility', 'control_precision', 'deductive_reasoning', 'depth_perception', 'dynamic_flexibility', 'dynamic_strength', 'explosive_strength', 'extent_flexibility', 'far_vision', 'finger_dexterity', 'flexibility_of_closure', 'fluency_of_ideas', 'glare_sensitivity', 'gross_body_coordination', 'gross_body_equilibrium', 'hearing_sensitivity', 'inductive_reasoning', 'information_ordering', 'manual_dexterity', 'mathematical_reasoning', 'memorization', 'multilimb_coordination', 'near_vision', 'night_vision', 'number_facility', 'oral_comprehension', 'oral_expression', 'originality', 'perceptual_speed', 'peripheral_vision', 'problem_sensitivity', 'rate_control', 'reaction_time', 'response_orientation', 'selective_attention', 'sound_localization', 'spatial_orientation', 'speech_clarity', 'speech_recognition', 'speed_of_closure', 'speed_of_limb_movement', 'stamina', 'static_strength', 'time_sharing', 'trunk_strength', 'visual_co

In [14]:
import os

# ── Fix duplicate title columns ──────────────────────────────────
# OCC_TITLE_x is from O*NET, OCC_TITLE_y is from BLS — keep O*NET version
if 'OCC_TITLE_x' in master.columns:
    master = master.rename(columns={'OCC_TITLE_x': 'OCC_TITLE'})
if 'OCC_TITLE_y' in master.columns:
    master = master.drop(columns=['OCC_TITLE_y'])

# ── Final checks ─────────────────────────────────────────────────
print(f"Shape:            {master.shape}")
print(f"Unique SOC codes: {master['OCC_CODE'].nunique()}")
print(f"Missing values:   {master.isnull().sum().sum()}")

# ── Spot checks ──────────────────────────────────────────────────
print("\nSpot check — Chief Executives (11-1011):")
print(master[master['OCC_CODE'] == '11-1011']
      [['OCC_CODE', 'OCC_TITLE', 'AIOE',
        'TOT_EMP_2019', 'TOT_EMP_2024',
        'real_median_wage_2019', 'real_median_wage_2024',
        'emp_change_pct_19_24', 'wage_change_pct_19_24']].to_string())

print("\nSpot check — Registered Nurses (29-1141):")
print(master[master['OCC_CODE'] == '29-1141']
      [['OCC_CODE', 'OCC_TITLE', 'AIOE',
        'TOT_EMP_2019', 'TOT_EMP_2024',
        'emp_change_pct_19_24', 'wage_change_pct_19_24']].to_string())

print("\nTop 5 highest AIOE in master:")
print(master.nlargest(5, 'AIOE')[['OCC_CODE', 'OCC_TITLE', 'AIOE']].to_string())

print("\nBottom 5 lowest AIOE in master:")
print(master.nsmallest(5, 'AIOE')[['OCC_CODE', 'OCC_TITLE', 'AIOE']].to_string())

# ── Save ─────────────────────────────────────────────────────────
os.makedirs('data/processed', exist_ok=True)
master.to_csv('data/processed/master_occupations.csv', index=False)
print(f"\nSaved: data/processed/master_occupations.csv")

Shape:            (655, 122)
Unique SOC codes: 655
Missing values:   68

Spot check — Chief Executives (11-1011):
  OCC_CODE         OCC_TITLE      AIOE  TOT_EMP_2019  TOT_EMP_2024  real_median_wage_2019  real_median_wage_2024  emp_change_pct_19_24  wage_change_pct_19_24
0  11-1011  Chief Executives  1.334246      205890.0      211850.0              214526.98               206420.0               2.89475              -3.779003

Spot check — Registered Nurses (29-1141):
    OCC_CODE          OCC_TITLE     AIOE  TOT_EMP_2019  TOT_EMP_2024  emp_change_pct_19_24  wage_change_pct_19_24
242  29-1141  Registered Nurses  0.22941     2982280.0     3282010.0             10.050364               9.797426

Top 5 highest AIOE in master:
    OCC_CODE                                   OCC_TITLE      AIOE
263  29-9092                          Genetic Counselors  1.527667
48   13-2061                         Financial Examiners  1.526064
53   15-2011                                   Actuaries  1.516474
